# IBS Feature Calculation

## Import Libraries

In [2]:
# Import libraries
import pandas as pd
import datetime
import re

## Import Data

In [3]:
# Read the compiled reports and create each sheet as a dataframe
file_name = "../../Random_Generation/Outputs/compiled_IBS_reports.xlsx"

req_criteria_df = pd.read_excel(file_name, sheet_name='Request Criteria')
search_rec_df = pd.read_excel(file_name, sheet_name='Search Record')
current_business_df = pd.read_excel(file_name,  sheet_name='Current Business')
previous_business_df = pd.read_excel(file_name, sheet_name='Previous Business')
curr_business_lit_trace_df = pd.read_excel(file_name, sheet_name='Current Business Lit Trace')
business_lit_search_df = pd.read_excel(file_name, sheet_name='Business Lit Search')
bankruptcy_rec_df = pd.read_excel(file_name, sheet_name='Bankruptcy Record')
indv_lit_search_df = pd.read_excel(file_name, sheet_name='Individual Lit Search')
accreditation_df = pd.read_excel(file_name, sheet_name='Accreditation')
fin_rel_search_count_df = pd.read_excel(file_name, sheet_name='Financial Related Search Count')
comm_rel_search_count_df = pd.read_excel(file_name, sheet_name='Commercial Related Search Count')
leg_rel_search_count_df = pd.read_excel(file_name, sheet_name='Legal Related Search Count')

In [4]:
# Preview Request Criteria dataframe
req_criteria_df.head()

,request_date,requestor_name,requested_individual_name,requested_id_no,request_id
0,2023/02/17,SUBSCRIBER,SARA HOW XINYING,A200XXXXK,IBS SAMPLE
1,2023/03/05,SUBSCRIBER,SUSAN LEE XINYING,A215XXXXQ,IBS SAMPLE
2,2023/01/27,SUBSCRIBER,SARA BIN XINYING,A225XXXXL,IBS SAMPLE
3,2023/02/05,SUBSCRIBER,KEVIN SOUZ XINYING,A277XXXXE,IBS SAMPLE
4,2023/02/18,SUBSCRIBER,MAY MEI ABEL,A281XXXXF,IBS SAMPLE


## Feature Calculation (on one IBS report)

In [5]:
# Test on one IBS report
entity_no = current_business_df["registration_no"].iloc[0]
owner_id = current_business_df["entity_id"].iloc[0]
searchYear = 2023

In [6]:
# Preview Current Business dataframe
current_business_df.head()

,entity_id,business_name,registration_no,status,position,appointment_date,current_directors_todate_count,current_secretaries_todate_count,current_alternate_directors_todate_count,active_charges_todate_count,current_shareholders_todate_count,subsidiary_todate_count
0,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,72176074Z,CEASED REGISTRATION,SECRETARY,2020/10/27,3,0,3,3,1,1
1,A200XXXXK,MOONLIGHT GREENYARN REGIONAL CORP,T13HS5717E,CONVERTED TO LLP,DIRECTOR,2015/08/29,0,1,1,1,2,2
2,A200XXXXK,MOONLIGHT RIVER PREFECTURE CORP,T13FM3117Q,SUSPENDED,SECRETARY,2022/11/02,2,3,0,1,0,3
3,A215XXXXQ,TWILIGHT WATER REGIONAL CORP,61259165R,GAZETTED TO BE STRUCK OFF,DIRECTOR,2019/01/10,1,2,1,1,0,2
4,A215XXXXQ,SUNSET AIR INTERNATIONAL ENTERPRISE,T18CC0038D,NaN,SHAREHOLDER \nSECRETARY \nDIRECTOR \nALTERNATE...,2022/09/14,1,2,1,1,1,1


In [7]:
# Replace N/A status with ""
current_business_df["status"].fillna("", inplace=True)

In [8]:
# Calculate number of current live, liquidated and dissolved businesses
numCurrentLiveBusinesses = len(current_business_df[(current_business_df["status"].str.contains("LIVE")) &
                                                   (current_business_df["entity_id"]==owner_id)]["status"])

numCurrentLiquidatedBusinesses = len(current_business_df[(current_business_df["status"].str.contains("LIQUIDAT")) &
                                                         (current_business_df["entity_id"]==owner_id)]["status"])

numCurrentDissolvedBusinesses = len(current_business_df[(current_business_df["status"].str.contains("DISSOLV")) &
                                                         (current_business_df["entity_id"]==owner_id)]["status"])

print("numCurrentLiveBusinesses: ", numCurrentLiveBusinesses)
print("numCurrentLiquidatedBusinesses: ", numCurrentLiquidatedBusinesses)
print("numCurrentDissolvedBusinesses: ", numCurrentDissolvedBusinesses)

numCurrentLiveBusinesses:  0
numCurrentLiquidatedBusinesses:  0
numCurrentDissolvedBusinesses:  0


In [9]:
# Calculate number of key positions, active charges and subsidiaries in the business
numDirectors = current_business_df[current_business_df["registration_no"]==entity_no]["current_directors_todate_count"][0]
numAlternateDirectors = current_business_df[current_business_df["registration_no"]==entity_no]["current_alternate_directors_todate_count"][0]
numShareholders = current_business_df[current_business_df["registration_no"]==entity_no]["current_shareholders_todate_count"][0]
numSecretaries = current_business_df[current_business_df["registration_no"]==entity_no]["current_secretaries_todate_count"][0]
numActiveCharges = current_business_df[current_business_df["registration_no"]==entity_no]["active_charges_todate_count"][0]
numSubsidiary = current_business_df[current_business_df["registration_no"]==entity_no]["subsidiary_todate_count"][0]
numKeyPositions = numDirectors + numAlternateDirectors + numShareholders + numSecretaries

print("numDirectors: ", numDirectors) 
print("numAlternateDirectors: ", numAlternateDirectors)
print("numShareholders: ", numShareholders)
print("numSecretaries: ", numSecretaries)
print("numActiveCharges: ", numActiveCharges)
print("numSubsidiary: ", numSubsidiary)
print("numKeyPositions: ", numKeyPositions)

numDirectors:  3
numAlternateDirectors:  3
numShareholders:  1
numSecretaries:  0
numActiveCharges:  3
numSubsidiary:  1
numKeyPositions:  7


In [10]:
# Calculate number of current key positions, active charges and subsidiaries in the business compared to other businesses owned by the same owner 
numDirectorsAgainstNumOtherBusinessDirectors = numDirectors - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["current_directors_todate_count"].sum()
numAlternateDirectorsAgainstNumOtherBusinessAlternateDirectors = numAlternateDirectors - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["current_alternate_directors_todate_count"].sum()
numShareholdersAgainstNumOtherBusinessShareholders = numShareholders - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["current_shareholders_todate_count"].sum()
numSecretariesAgainstNumOtherBusinessSecretaries = numSecretaries - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["current_secretaries_todate_count"].sum()
numActiveChargesAgainstNumOtherBusinessActiveCharges = numActiveCharges - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["active_charges_todate_count"].sum()
numSubsidiaryAgainstNumOtherBusinessSubsidiary = numSubsidiary - current_business_df[(current_business_df["registration_no"]!=entity_no) & (current_business_df["entity_id"]==owner_id)]["subsidiary_todate_count"].sum()
numKeyPositionsAgainstNumOtherBusinessKeyPositions = numDirectorsAgainstNumOtherBusinessDirectors + numAlternateDirectorsAgainstNumOtherBusinessAlternateDirectors + numShareholdersAgainstNumOtherBusinessShareholders + numSecretariesAgainstNumOtherBusinessSecretaries

print("numDirectorsAgainstNumOtherBusinessDirectors: ", numDirectorsAgainstNumOtherBusinessDirectors) 
print("numAlternateDirectorsAgainstNumOtherBusinessAlternateDirectors: ", numAlternateDirectorsAgainstNumOtherBusinessAlternateDirectors)
print("numShareholdersAgainstNumOtherBusinessShareholders: ", numShareholdersAgainstNumOtherBusinessShareholders)
print("numSecretariesAgainstNumOtherBusinessSecretaries: ", numSecretariesAgainstNumOtherBusinessSecretaries)
print("numActiveChargesAgainstNumOtherBusinessActiveCharges: ", numActiveChargesAgainstNumOtherBusinessActiveCharges)
print("numSubsidiaryAgainstNumOtherBusinessSubsidiary: ", numSubsidiaryAgainstNumOtherBusinessSubsidiary)
print("numKeyPositionsAgainstNumOtherBusinessKeyPositions: ", numKeyPositionsAgainstNumOtherBusinessKeyPositions)

numDirectorsAgainstNumOtherBusinessDirectors:  1
numAlternateDirectorsAgainstNumOtherBusinessAlternateDirectors:  2
numShareholdersAgainstNumOtherBusinessShareholders:  -1
numSecretariesAgainstNumOtherBusinessSecretaries:  -4
numActiveChargesAgainstNumOtherBusinessActiveCharges:  1
numSubsidiaryAgainstNumOtherBusinessSubsidiary:  -4
numKeyPositionsAgainstNumOtherBusinessKeyPositions:  -2


In [11]:
# Calculate whether current appointment in the business is longest or most recent
isLongestAppointment = current_business_df[(current_business_df["entity_id"]==owner_id)].sort_values(by='appointment_date', ascending=True).iloc[0]["registration_no"] == entity_no
isMostRecentAppointment = current_business_df[(current_business_df["entity_id"]==owner_id)].sort_values(by='appointment_date', ascending=False).iloc[0]["registration_no"] == entity_no

print("isLongestAppointment: ", isLongestAppointment)
print("isMostRecentAppointment: ", isMostRecentAppointment)

isLongestAppointment:  False
isMostRecentAppointment:  False


In [12]:
# Retrieve the business entity type from the UEN of the business 

mappingDF = pd.read_csv("../../Dataset/uen_data/UEN Mapping.csv")

def entityTypeMapping(uen):
    """
    Maps a 9-character or 10-character UEN number to an entity-type 
    in accordance to https://www.uen.gov.sg/ueninternet/faces/pages/admin/aboutUEN.jspx
    """

    if len(uen) == 9: 
        # business registered with ACRA
        return "ACRA-registered Business"
    elif len(uen) == 10 and uen[3].isnumeric(): 
        # local company registered with ACRA
        return "ACRA-registered Local Company"
    elif len(uen) == 10 and not (uen[3].isnumeric()):
        # all other entities
        return mappingDF[mappingDF["Entity-Type Indicator"]==uen[3:5]]["Entity-Type"].iloc[0]
    else:
        return "Invalid UEN"


entityType = entityTypeMapping( current_business_df[(current_business_df["registration_no"]==entity_no)]["registration_no"][0] )

print("entityType: ", entityType)

entityType:  ACRA-registered Business


In [13]:
# Preview Previous Business dataframe
previous_business_df.head()

,entity_id,business_name,registration_no,status,position,cessation_date
0,A200XXXXK,DUSK AIR INTERNATIONAL ENTERPRISE,201302288A,LIVE,SHAREHOLDER,1989/12/04
1,A200XXXXK,MIDNIGHT RIVER PREFECTURE CORP,201384994K,STRUCK OFF,SHAREHOLDER,2012/01/09
2,A215XXXXQ,MIDNIGHT WATER STATE CORP,201834753O,DISSOLVED,ALTERNATE DIRECTOR,1906/04/22
3,A215XXXXQ,SUNSET WOOD PREFECTURE ENTERPRISE,T18LL6107S,CONVERTED TO LLP,SHAREHOLDER,1962/07/24
4,A215XXXXQ,DAWN AIR CENTRAL ENTERPRISE,14792205S,CANCELLED,SECRETARY,2000/01/31


In [14]:
# Calculate number of previously terminated and cancelled businesses
numPreviousTerminatedBusinesses = len(previous_business_df[(previous_business_df["status"].str.contains("TERMINAT")) & (previous_business_df["entity_id"]==owner_id)])
numPreviousCancelledBusinesses = len(previous_business_df[(previous_business_df["status"].str.contains("CANCEL")) & (previous_business_df["entity_id"]==owner_id)])

print("numPreviousTerminatedBusinesses: ", numPreviousTerminatedBusinesses)
print("numPreviousCancelledBusinesses: ", numPreviousCancelledBusinesses)

numPreviousTerminatedBusinesses:  0
numPreviousCancelledBusinesses:  0


In [15]:
# Preview Current Business Litigation Trace dataframe
curr_business_lit_trace_df.head()

,entity_id,business_name,year,enquiries,current_year_after_9_Jan,previous_year_before_9_Jan
0,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,2022,"Litigation Trace: District Court \n($60,000 - ...",0,0
1,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,2022,"Litigation Trace: High Court \n(> $250,000)",0,1
2,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,2022,"Litigation Trace: Magistrate Court \n(< $60,000)",1,0
3,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,2022,OS/OA/OP/OM Trace,0,0
4,A200XXXXK,SUNSET WOOD INTERNATIONAL ENTERPRISE,2022,Winding Up Trace,0,1


In [16]:
# Calculate total number and proportion of each litigation type
numMCLitigation = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["enquiries"]=="Litigation Trace: Magistrate Court\n(< $60,000)") &
     (curr_business_lit_trace_df["year"]==searchYear)
]["current_year_after_9_Jan"].sum()

numDCLitigation = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["enquiries"]=="Litigation Trace: District Court\n($60,000 - $250,000)") & 
     (curr_business_lit_trace_df["year"]==searchYear)
]["current_year_after_9_Jan"].sum()

numHCLitigation = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["enquiries"]=="Litigation Trace: High Court\n(> $250,000)") &
     (curr_business_lit_trace_df["year"]==searchYear)
]["current_year_after_9_Jan"].sum()

numWindingUpLitigation = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["enquiries"]=="Winding Up Trace") &
     (curr_business_lit_trace_df["year"]==searchYear)
]["current_year_after_9_Jan"].sum()

numOSOAOPOMLitigation = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["enquiries"]=="OS/OA/OP/OM Trace") &
     (curr_business_lit_trace_df["year"]==searchYear)
]["current_year_after_9_Jan"].sum()

numLitigationTraceBusinesses = curr_business_lit_trace_df[
     (curr_business_lit_trace_df["entity_id"]==owner_id) &
     (curr_business_lit_trace_df["year"]==searchYear)
]["business_name"].nunique()

if (numLitigationTraceBusinesses != 0):
     avgNumMCLitigation = numMCLitigation / numLitigationTraceBusinesses
     avgNumDCLitigation = numDCLitigation / numLitigationTraceBusinesses
     avgNumHCLitigation = numHCLitigation / numLitigationTraceBusinesses
     avgNumWindingUpLitigation = numWindingUpLitigation / numLitigationTraceBusinesses
     avgNumOSOAOPOMLitigation = numOSOAOPOMLitigation / numLitigationTraceBusinesses
else:
     avgNumMCLitigation = 0
     avgNumDCLitigation = 0
     avgNumHCLitigation = 0
     avgNumWindingUpLitigation = 0
     avgNumOSOAOPOMLitigation = 0

print("numMCLitigation: ", numMCLitigation)
print("numDCLitigation: ", numDCLitigation)
print("numHCLitigation: ", numHCLitigation)
print("numWindingUpLitigation: ", numWindingUpLitigation)
print("numOSOAOPOMLitigation: ", numOSOAOPOMLitigation)
print("numLitigationTraceBusinesses: ", numLitigationTraceBusinesses)
print("avgNumMCLitigation: ", avgNumMCLitigation)
print("avgNumDCLitigation: ", avgNumDCLitigation)
print("avgNumHCLitigation: ", avgNumHCLitigation)
print("avgNumWindingUpLitigation: ", avgNumWindingUpLitigation)
print("avgNumOSOAOPOMLitigation: ", avgNumOSOAOPOMLitigation)

numMCLitigation:  0
numDCLitigation:  0
numHCLitigation:  0
numWindingUpLitigation:  0
numOSOAOPOMLitigation:  0
numLitigationTraceBusinesses:  0
avgNumMCLitigation:  0
avgNumDCLitigation:  0
avgNumHCLitigation:  0
avgNumWindingUpLitigation:  0
avgNumOSOAOPOMLitigation:  0


In [17]:
# Pre-process and preview Business Litigation Search dataframe
business_lit_search_df['amount'] = list(map(lambda x: re.findall(r'\d+', x), business_lit_search_df['amount']))
business_lit_search_df['amount'] = list(map(lambda x: sum(list(map(lambda y: float(y), x))), business_lit_search_df['amount']))
business_lit_search_df["filing_date"] = pd.to_datetime(business_lit_search_df["filing_date"])
business_lit_search_df["months_since_filing"] = list(map(lambda x: (datetime.date.today().year - x.year)*12 + datetime.date.today().month - x.month, business_lit_search_df["filing_date"]))
business_lit_search_df["entity_type"] = list(map(lambda x: 'Company' if 'LIMITED' in x else 'Individual', business_lit_search_df['plaintiff']))
business_lit_search_df.head()

,entity_id,sn,court_type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,...,amount,last_update,main_cause,individual_id,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category,months_since_filing,entity_type
0,A200XXXXK,1,DC / WOS,7556,2014,2018-07-24,MIDNIGHT AIR INTERNATIONAL PTE LIMITED,CONCLUDED (16-OCT-2009),2004/12/24,NaN,...,86754.0,2010/02/09,Others(By a Loan Agreement dated the 13th day ...,A200XXXXK,Financials,Concluded,Discharge Notice,Others,57,Company
1,A215XXXXQ,1,MC / WOS,1387,2021,2017-09-15,MOONLIGHT RIVER CENTRAL PTE LIMITED,CONCLUDED (23-AUG-2008),2000/07/09,NaN,...,22453.0,2001/01/14,Contract-Others(Breach of Loan Agreements.),A215XXXXQ,Financials,Concluded,Bankruptcy Related,Annulment,67,Company
2,A215XXXXQ,2,SC / OC,2402,2019,2007-01-17,SUNSET PLASTIC REGIONAL CORP,WITHDRAWN WITH LIBERTY TO RESTORE,2018/07/09,NaN,...,32137.0,2013/11/22,Contract-Others(Sub-contractual works),A215XXXXQ,Contracts,Others,Judgement,Judgement,195,Individual
3,A215XXXXQ,3,SC / WOS,9940,2019,2009-05-17,SUNSET RIVER STATE PTE LIMITED,SERVICE OF THE WRIT OF SUMMONS,2002/02/09,NaN,...,76614.0,2010/06/25,Others(Outstanding Licence Fee),A215XXXXQ,Financials,Others,Judgement,Others,167,Company
4,A215XXXXQ,4,SC / WOS,6633,2021,2016-03-10,DUSK GREENYARN REGIONAL ENTERPRISE,CONCLUDED (11-DEC-2013),2004/11/11,NaN,...,79523.0,2002/09/24,Others(Restitution - Repayment of monies misap...,A215XXXXQ,Others,Concluded,Others,Others,85,Individual


In [18]:
# Calculate business litigation counts and amounts across court types, number of months, entity type, status, and cause
for court_type in ["MC / WOS", "DC / WOS", "SC / WOS", "MC / OC", "DC / OC", "SC / OC"]:
     for num_months in [3, 6, 9, 12, 24, 36]:
          for entity_type in ["Company", "Individual"]:
               for status in ["Concluded", "Bankruptcy Related", "Discharged", "Others"]:
                    for cause in ["Negligence & Accidents", "Financials", "Contracts", "Provision of Goods and Services", "Others"]:

                         business_lit_search_df[
                              (business_lit_search_df["entity_id"]==owner_id) &
                              (business_lit_search_df["court_type"] == court_type) &
                              (business_lit_search_df["months_since_filing"] < num_months) &
                              (business_lit_search_df["entity_type"] == entity_type) &
                              (business_lit_search_df["case_status_category"] == status) &
                              (business_lit_search_df["main_cause_category"] == cause) 
                         ]["amount"].sum()

                         business_lit_search_df[
                              (business_lit_search_df["entity_id"]==owner_id) &
                              (business_lit_search_df["court_type"] == court_type) &
                              (business_lit_search_df["months_since_filing"] < num_months) &
                              (business_lit_search_df["entity_type"] == entity_type) &
                              (business_lit_search_df["case_status_category"] == status) &
                              (business_lit_search_df["main_cause_category"] == cause) 
                         ]["amount"].count()

In [19]:
# Pre-process and preview Bankruptcy Records dataframe
bankruptcy_rec_df['amount'] = list(map(lambda x: re.findall(r'\d+', x), bankruptcy_rec_df['amount']))
bankruptcy_rec_df['amount'] = list(map(lambda x: sum(list(map(lambda y: float(y), x))), bankruptcy_rec_df['amount']))
bankruptcy_rec_df["filing_date"] = pd.to_datetime(bankruptcy_rec_df["filing_date"])
bankruptcy_rec_df["months_since_filing"] = list(map(lambda x: (datetime.date.today().year - x.year)*12 + datetime.date.today().month - x.month, bankruptcy_rec_df["filing_date"]))
bankruptcy_rec_df["entity_type"] = list(map(lambda x: 'Company' if 'LIMITED' in x else 'Individual', bankruptcy_rec_df['plaintiff']))
bankruptcy_rec_df.head()

,entity_id,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,plaintiff_status,plaintiff_status_date,defendant_status,defendant_status_date,amount,last_update,name_of_private_trustees,case_status_category,plaintiff_status_category,defendant_status_category,months_since_filing,entity_type
0,A200XXXXK,611,2015,2012-06-10,SUNSET GREENYARN REGIONAL ENTERPRISE,CONCLUDED (09-MAR-2010),2005/02/12,NaN,DISCONTINUED/WITHDRAW,2004/11/03,UNDISCHARGED BANKRUPT,2005/02/20,58832.0,2014/09/01,MAY SOUZ PATHERSON CALDECOTT BOULEVARD #26-28 ...,Concluded,Others,Bankruptcy Related,130,Individual
1,A200XXXXK,461,2014,2014-04-05,MOONLIGHT RIVER INTERNATIONAL PTE LIMITED,CONCLUDED (27-FEB-2013),2013/02/27,NaN,DEFAULT JUDGMENT -O13,2018/03/05,CERTIFICATE OF ANNULMENT BY OA UNDER S95A,2017/05/09,65305.0,2007/12/22,MABLE LEE ABEL CHANGI BROADWAY #26-2 SINGAPORE...,Concluded,Judgement,Annulment,108,Company
2,A200XXXXK,4332,2019,2010-02-14,DAWN METAL CENTRAL ENTERPRISE,CONCLUDED (09-JUN-2010),2004/06/17,NaN,PETITION FILED,2007/11/23,"SETTLED -AFTER SET DOWN, BEFORE TRIAL",2000/08/16,31982.0,2002/02/04,CHARLES LIN ABEL PASIR RIS AVENUE #9-17 SINGAP...,Concluded,Others,Judgement,158,Individual
3,A215XXXXQ,3733,2019,2007-03-22,SUNSET WOOD INTERNATIONAL ENTERPRISE,FOR DISCHARGE,2019/02/13,NaN,CERTIFICATE OF ANNULMENT BY OA UNDER S95A,2000/09/18,PETITION DISMISSED,2018/03/19,72060.0,2007/09/01,TOM LIN NAQUILA PUNGGOL DRIVE #18-27 SINGAPORE...,Discharged,Annulment,Others,193,Individual
4,A215XXXXQ,2902,2020,2012-05-29,SUNSET METAL REGIONAL CORP,CONCLUDED (07-NOV-2013),2012/05/07,NaN,OTHER JUDGMENT,2013/02/21,JUDGMENT 13,2000/11/26,80250.0,2005/05/14,MAY LIN NAQUILA WESTWORD MAISON #1-48 SINGAPOR...,Concluded,Judgement,Judgement,131,Individual


In [20]:
# Calculate bankruptcy record counts and amounts across number of months, entity type, and status
for num_months in [3, 6, 9, 12, 24, 36]:
     for entity_type in ["Company", "Individual"]:
          for status in ["Concluded", "Bankruptcy Related", "Discharged", "Others"]:

                    bankruptcy_rec_df[
                         (bankruptcy_rec_df["entity_id"]==owner_id) &
                         (bankruptcy_rec_df["months_since_filing"] < num_months) &
                         (bankruptcy_rec_df["entity_type"] == entity_type) &
                         (bankruptcy_rec_df["case_status_category"] == status) 
                    ]["amount"].sum()

                    bankruptcy_rec_df[
                         (bankruptcy_rec_df["entity_id"]==owner_id) &
                         (bankruptcy_rec_df["months_since_filing"] < num_months) &
                         (bankruptcy_rec_df["entity_type"] == entity_type) &
                         (bankruptcy_rec_df["case_status"] == status) 
                    ]["amount"].count()

In [21]:
# Pre-process and preview Individual Litigation Search dataframe
indv_lit_search_df['amount'] = list(map(lambda x: re.findall(r'\d+', x), indv_lit_search_df['amount']))
indv_lit_search_df['amount'] = list(map(lambda x: sum(list(map(lambda y: float(y), x))), indv_lit_search_df['amount']))
indv_lit_search_df["filing_date"] = pd.to_datetime(indv_lit_search_df["filing_date"])
indv_lit_search_df["months_since_filing"] = list(map(lambda x: (datetime.date.today().year - x.year)*12 + datetime.date.today().month - x.month, indv_lit_search_df["filing_date"]))
indv_lit_search_df["entity_type"] = list(map(lambda x: 'Company' if 'LIMITED' in x else 'Individual', indv_lit_search_df['plaintiff']))
indv_lit_search_df.head()

,entity_id,sn,court_type,case_no,year,filing_date,plaintiff,case_status,case_status_date,case_status_remarks,...,defendant_status_date,amount,last_update,main_cause,main_cause_category,case_status_category,plaintiff_status_category,defendant_status_category,months_since_filing,entity_type
0,A200XXXXK,1,SC / WOS,7788,2016,2019-08-02,SUSAN MEI LEQUAN,CONCLUDED (21-APR-2011),2012/09/22,NaN,...,2000/06/20,81843.0,2010/01/23,Contract-Others(Breach of Contract for the ren...,Contracts,Concluded,Annulment,Others,44,Individual
1,A200XXXXK,2,SC / WOS,2463,2016,2011-01-12,SUSAN SOUZ XINYING,PETITION DISMISSED,2005/07/15,NaN,...,2015/12/11,68526.0,2009/06/30,Contract-Others(Building and Construction),Provision of Goods and Services,Others,Others,Others,147,Individual
2,A200XXXXK,3,SC / WOS,3661,2014,2002-08-05,GOTHRY BIN PATHERSON,CONCLUDED-CONSOLIDATED,2011/08/03,NaN,...,2006/12/06,63013.0,2010/11/20,Contract-Others(personal loan),Financials,Concluded,Discharge Notice,Others,248,Individual
3,A200XXXXK,4,DC / OC,3871,2016,2008-05-12,GOTHRY NG ABEL,CONCLUDED (22-FEB-2005),2016/10/12,NaN,...,2018/04/18,36407.0,2003/12/17,Contract-Others(Sale and purchase of shares in...,Contracts,Concluded,Others,Others,179,Individual
4,A215XXXXQ,1,MC / WOS,7787,2020,2013-01-26,SUSAN LIN XINYING,CONCLUDED (24-NOV-2009),2004/03/07,NaN,...,2009/04/05,77655.0,2011/02/20,Contract-Others(Breach and Termination of Dist...,Contracts,Concluded,Others,Bankruptcy Related,123,Individual


In [22]:
# Calculate individual litigation counts and amounts across court types, number of months, entity type, status, and cause
for court_type in ["MC / WOS", "DC / WOS", "SC / WOS", "MC / OC", "DC / OC", "SC / OC"]:
     for num_months in [3, 6, 9, 12, 24, 36]:
          for entity_type in ["Company", "Individual"]:
               for status in ["Concluded", "Bankruptcy Related", "Discharged", "Others"]:
                    for cause in ["Negligence & Accidents", "Financials", "Contracts", "Provision of Goods and Services", "Others"]:

                         indv_lit_search_df[
                              (indv_lit_search_df["entity_id"]==owner_id) &
                              (indv_lit_search_df["court_type"] == court_type) &
                              (indv_lit_search_df["months_since_filing"] < num_months) &
                              (indv_lit_search_df["entity_type"] == entity_type) &
                              (indv_lit_search_df["case_status_category"] == status) &
                              (indv_lit_search_df["main_cause_category"] == cause) 
                         ]["amount"].sum()

                         indv_lit_search_df[
                              (indv_lit_search_df["entity_id"]==owner_id) &
                              (indv_lit_search_df["court_type"] == court_type) &
                              (indv_lit_search_df["months_since_filing"] < num_months) &
                              (indv_lit_search_df["entity_type"] == entity_type) &
                              (indv_lit_search_df["case_status_category"] == status) &
                              (indv_lit_search_df["main_cause_category"] == cause) 
                         ]["amount"].count()

In [23]:
# Preview Financial Related Search Count dataframe
fin_rel_search_count_df.head()

,entity_id,year,month,purpose,count
0,A200XXXXK,2022,All,All,3
1,A200XXXXK,2022,Jan,All,0
2,A200XXXXK,2022,Feb,All,0
3,A200XXXXK,2022,Mar,All,0
4,A200XXXXK,2022,Apr,All,1


In [24]:
# Preview Commercial Related Search Count dataframe
comm_rel_search_count_df.head()

,entity_id,year,month,purpose,count
0,A200XXXXK,2022,All,All,2
1,A200XXXXK,2022,Jan,All,1
2,A200XXXXK,2022,Feb,All,0
3,A200XXXXK,2022,Mar,All,0
4,A200XXXXK,2022,Apr,All,0


In [25]:
# Preview Legal Related Search Count dataframe
leg_rel_search_count_df.head()

,entity_id,year,month,purpose,count
0,A200XXXXK,2022,All,All,4
1,A200XXXXK,2022,Jan,All,0
2,A200XXXXK,2022,Feb,All,0
3,A200XXXXK,2022,Mar,All,0
4,A200XXXXK,2022,Apr,All,1


In [26]:
# Calculate recent year financial, commercial and legal search counts
recentYearIndividualFinancialSearchCount = fin_rel_search_count_df[(fin_rel_search_count_df["entity_id"]==owner_id) &
                                                                   (fin_rel_search_count_df["purpose"]=="All") &
                                                                   (fin_rel_search_count_df["year"]==searchYear-1) & 
                                                                   (fin_rel_search_count_df["month"]=="All")
                                                                   ]["count"].sum()

recentYearIndividualTradeInsuranceSearchCount = fin_rel_search_count_df[(fin_rel_search_count_df["entity_id"]==owner_id) &
                                                                        (fin_rel_search_count_df["purpose"]=="Trade Insurance") &
                                                                        (fin_rel_search_count_df["year"]==searchYear-1) & 
                                                                        (fin_rel_search_count_df["month"]=="All")
                                                                        ]["count"].sum()

recentYearIndividualFinancialInstitutionSearchCount = fin_rel_search_count_df[(fin_rel_search_count_df["entity_id"]==owner_id) &
                                                                              (fin_rel_search_count_df["purpose"]=="Financial Institution") &
                                                                              (fin_rel_search_count_df["year"]==searchYear-1) & 
                                                                              (fin_rel_search_count_df["month"]=="All")
                                                                              ]["count"].sum()

recentYearIndividualHirePurchaseSearchCount = fin_rel_search_count_df[(fin_rel_search_count_df["entity_id"]==owner_id) &
                                                                      (fin_rel_search_count_df["purpose"]=="Hire Purchase / Leasing") &
                                                                      (fin_rel_search_count_df["year"]==searchYear-1) & 
                                                                      (fin_rel_search_count_df["month"]=="All")
                                                                      ]["count"].sum()

recentYearIndividualCommercialSearchCount = comm_rel_search_count_df[(comm_rel_search_count_df["entity_id"]==owner_id) &
                                                                     (comm_rel_search_count_df["purpose"]=="All") &
                                                                     (comm_rel_search_count_df["year"]==searchYear-1) & 
                                                                     (comm_rel_search_count_df["month"]=="All")
                                                                     ]["count"].sum()

recentYearIndividualCommercialRetailSearchCount = comm_rel_search_count_df[(comm_rel_search_count_df["entity_id"]==owner_id) &
                                                                           (comm_rel_search_count_df["purpose"]=="Commercial-Retail") &
                                                                           (comm_rel_search_count_df["year"]==searchYear-1) & 
                                                                          (comm_rel_search_count_df["month"]=="All")
                                                                           ]["count"].sum()

recentYearIndividualServicesSearchCount = comm_rel_search_count_df[(comm_rel_search_count_df["entity_id"]==owner_id) &
                                                                           (comm_rel_search_count_df["purpose"]=="Services") &
                                                                           (comm_rel_search_count_df["year"]==searchYear-1) & 
                                                                           (comm_rel_search_count_df["month"]=="All")
                                                                           ]["count"].sum()

recentYearIndividualLegalSearchCount = leg_rel_search_count_df[(leg_rel_search_count_df["entity_id"]==owner_id) &
                                                               (leg_rel_search_count_df["purpose"]=="All") &
                                                               (leg_rel_search_count_df["year"]==searchYear-1) & 
                                                               (leg_rel_search_count_df["month"]=="All")
                                                               ]["count"].sum()

recentYearIndividualLitigationSearchCount = leg_rel_search_count_df[(leg_rel_search_count_df["entity_id"]==owner_id) &
                                                                    (leg_rel_search_count_df["purpose"]=="Litigation") &
                                                                    (leg_rel_search_count_df["year"]==searchYear-1) & 
                                                                    (leg_rel_search_count_df["month"]=="All")
                                                                    ]["count"].sum()

recentYearIndividualConveyancingSearchCount = leg_rel_search_count_df[(leg_rel_search_count_df["entity_id"]==owner_id) &
                                                                      (leg_rel_search_count_df["purpose"]=="Conveyancing") &
                                                                      (leg_rel_search_count_df["year"]==searchYear-1) & 
                                                                      (leg_rel_search_count_df["month"]=="All")
                                                                      ]["count"].sum()

print("recentYearIndividualFinancialSearchCount: ", recentYearIndividualFinancialSearchCount)
print("recentYearIndividualTradeInsuranceSearchCount:", recentYearIndividualTradeInsuranceSearchCount)
print("recentYearIndividualFinancialInstitutionSearchCount: ", recentYearIndividualFinancialInstitutionSearchCount)
print("recentYearIndividualHirePurchaseSearchCount: ", recentYearIndividualHirePurchaseSearchCount)

print("recentYearIndividualCommercialSearchCount: ", recentYearIndividualCommercialSearchCount)
print("recentYearIndividualCommercialRetailSearchCount: ", recentYearIndividualCommercialRetailSearchCount)
print("recentYearIndividualServicesSearchCount: ", recentYearIndividualServicesSearchCount)

print("recentYearIndividualLegalSearchCount: ", recentYearIndividualLegalSearchCount)
print("recentYearIndividualLitigationSearchCount: ", recentYearIndividualLitigationSearchCount)
print("recentYearIndividualConveyancingSearchCount: ", recentYearIndividualConveyancingSearchCount)

recentYearIndividualFinancialSearchCount:  3
recentYearIndividualTradeInsuranceSearchCount: 0
recentYearIndividualFinancialInstitutionSearchCount:  3
recentYearIndividualHirePurchaseSearchCount:  0
recentYearIndividualCommercialSearchCount:  2
recentYearIndividualCommercialRetailSearchCount:  2
recentYearIndividualServicesSearchCount:  0
recentYearIndividualLegalSearchCount:  4
recentYearIndividualLitigationSearchCount:  0
recentYearIndividualConveyancingSearchCount:  4
